# Anime

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
import numpy as np
import plotly.graph_objects as go
import ast
import plotly.express as px

In [3]:
# load dataset
df_profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")
df_person_anime_works = pd.read_csv("../../data/cleaned/person_anime_works_cleaned.csv")
df_person_details = pd.read_csv("../../data/cleaned/person_details_cleaned.csv")
df_person_alternate_names = pd.read_csv("../../data/cleaned/person_alternate_names_cleaned.csv")



In [4]:
df_person_details["birthday"] = pd.to_datetime(df_person_details["birthday"], errors="coerce")
df_person_details["birth_year"] = df_person_details["birthday"].dt.year.astype("Int64")

df_person_details["favorites"] = pd.to_numeric(df_person_details["favorites"], errors="coerce")


## In quale aree del mondo gli anime sono maggiormente seguiti?


In [5]:
#Load data
profiles = pd.read_csv("../../data/cleaned/profiles_cleaned.csv")

# extract country from location
profiles["country"] = (
    profiles["location"]
    .astype("string")
    .str.strip()
    .replace({"": pd.NA})
    .str.split(",")
    .str[-1]
    .str.strip()
)

#country names
country_fix = {
    "Usa": "United States",
    "US": "United States",
    "United States Of America": "United States",
    "UK": "United Kingdom",
    "England": "United Kingdom",
    "Scotland": "United Kingdom",
    "Russian Federation": "Russia",
    "Russia": "Russia",
    "South Korea": "Korea, Rep." ,
    "Korea, Republic of": "South Korea",

}
profiles["country"] = profiles["country"].replace(country_fix)

profiles["country"] = profiles["country"].where(
    profiles["country"].isna(),
    profiles["country"].str.replace(r"\s+", " ", regex=True).str.title()
)
#unique users per country
country_counts = (
    profiles.dropna(subset=["country"])
            .groupby("country")["username"]
            .nunique()
            .reset_index(name="n_users")
)

#which countries are not recognised by Plotly?
valid_countries = px.data.gapminder()["country"].unique()
not_recognised = (
    country_counts.loc[~country_counts["country"].isin(valid_countries), ["country", "n_users"]]
    .sort_values("n_users", ascending=False)
)
not_recognised.head(30)

# log scale
country_counts["log_users"] = np.log10(country_counts["n_users"] + 1)

# choropleth map
fig = px.choropleth(
    country_counts,
    locations="country",
    locationmode="country names",
    color="log_users",
    hover_name="country",
    hover_data={"n_users": True, "log_users": ":.2f"},
    title="Dove gli anime sono maggiormente seguiti? (utenti per paese, scala log10)",
)

fig.update_layout(
    coloraxis_colorbar=dict(title="log10(utenti)", tickformat=".1f")
)

fig.update_geos(showland=True, landcolor="lightgrey", showocean=True, oceancolor="#e6f2ff")

#Layout
fig.update_layout(template="plotly_white", width=1100, height=600)
fig.show()


/var/folders/h7/9ssx0z652tq5bd60x8c9jc500000gn/T/ipykernel_13074/3942446973.py:55: DeprecationWarning: The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.
  fig = px.choropleth(
